### Import Dependencies

In [1]:
import random
from qdrant_client import QdrantClient
import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
import numpy as np
from qdrant_client import QdrantClient

### Fictional Warehouses

In [2]:
warehouses = [
    {
        "warehouse_id": "DE-BER-01",
        "warehouse_location": "Berlin, Germany",
        "warehouse_name": "Berlin Distribution Center"
    },
    {
        "warehouse_id": "DE-MUN-01",
        "warehouse_location": "Munich, Germany",
        "warehouse_name": "Munich Logistics Hub"
    },
    {
        "warehouse_id": "DE-HAM-01",
        "warehouse_location": "Hamburg, Germany",
        "warehouse_name": "Hamburg North Warehouse"
    },
    {
        "warehouse_id": "FR-PAR-01",
        "warehouse_location": "Paris, France",
        "warehouse_name": "Paris Central Depot"
    },
    {
        "warehouse_id": "FR-LYO-01",
        "warehouse_location": "Lyon, France",
        "warehouse_name": "Lyon Regional Warehouse"
    },
    {
        "warehouse_id": "FR-MAR-01",
        "warehouse_location": "Marseille, France",
        "warehouse_name": "Marseille Mediterranean Hub"
    }
]

### Simulate Stock Availability for each of the warehouses

#### Retrieve all item IDs from the Amamzon items Qdrant Collection

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
dummy_vector = np.zeros(1536).tolist()

In [5]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
).points

In [6]:
payload

[ScoredPoint(id=410, version=3, score=0.0, payload={'parent_asin': 'B08356M6N7'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=83, version=3, score=0.0, payload={'parent_asin': 'B0085CFFFA'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=363, version=3, score=0.0, payload={'parent_asin': 'B00B5HVEMK'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=45, version=3, score=0.0, payload={'parent_asin': 'B06X3WGFGK'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=425, version=3, score=0.0, payload={'parent_asin': 'B09Q29B8MG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=207, version=3, score=0.0, payload={'parent_asin': 'B081SPHPCL'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=67, version=3, score=0.0, payload={'parent_asin': 'B07YQW55CN'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=278, version=3, score=0.0, payload={'parent_asin': 'B007136D32'}, vector=None

In [7]:
len(payload)

1000

In [8]:
parent_asin_list = [item.payload["parent_asin"] for item in payload]

In [10]:
parent_asin_list

['B08356M6N7',
 'B0085CFFFA',
 'B00B5HVEMK',
 'B06X3WGFGK',
 'B09Q29B8MG',
 'B081SPHPCL',
 'B07YQW55CN',
 'B007136D32',
 'B07MG8WQKF',
 'B07FF1MK43',
 'B08LKK8CSD',
 'B001YQ74FI',
 'B07569NQ5X',
 'B009WYJBEG',
 'B096ZRXPKC',
 'B004NBY4AE',
 'B00TUGU9YC',
 'B005DPEZ4Y',
 'B00D9XJL00',
 'B0074CISLK',
 'B004VRKSSM',
 'B088FDM3K6',
 'B00W9MM21W',
 'B097GYG97Y',
 'B07JNCVN9G',
 'B00CXUT5ZG',
 'B07N66CXX5',
 'B076VJDWBP',
 'B07V718F4S',
 'B08S59RHPR',
 'B074KCJ926',
 'B07PFD5ZVS',
 'B08DK692G2',
 'B00IKMJOZC',
 'B08P4S4S4C',
 'B086SR45BY',
 'B00R46XWTE',
 'B07LFCYHWW',
 'B07Z8S8JTH',
 'B09L19PLFM',
 'B07J31M7PW',
 'B00WSP5M8A',
 'B09JKDLQMP',
 'B00SGDYHA2',
 'B09F6MJ6HX',
 'B07MK6DJ6M',
 'B000923HCC',
 'B004HYG9TG',
 'B09STBPJ53',
 'B00MTSWMV6',
 'B07ZCQTV1X',
 'B008HWTVQ2',
 'B08FKBXCMR',
 'B01MR77T7S',
 'B07Z4NG8J7',
 'B07BSPYLVG',
 'B074QJ7CFZ',
 'B0932Z5W3S',
 'B08FNNZTH1',
 'B07S7CT2L4',
 'B00RK7ODDG',
 'B076561VC1',
 'B09F2P5CKM',
 'B00DVTBM2W',
 'B0042TW3J6',
 'B088FV6FC4',
 'B07PNG48

### Generate Synthetic availability for all items in Qdrant

In [11]:
def generate_inventory_data(warehouses, product_ids, availability_rate=0.75):
    
    inventory_records = []
    
    for warehouse in warehouses:
        for product_id in product_ids:
            # 75% chance the product is available in this warehouse
            if random.random() < availability_rate:
                total_quantity = random.randint(0, 100)
                
                # Only add to inventory if quantity > 0
                if total_quantity > 0:
                    inventory_records.append({
                        "warehouse_id": warehouse["warehouse_id"],
                        "warehouse_location": warehouse["warehouse_location"],
                        "warehouse_name": warehouse["warehouse_name"],
                        "product_id": product_id,
                        "total_quantity": total_quantity,
                        "reserved_quantity": 0  # Starting with no reservations
                    })
    
    return inventory_records

In [12]:
inventory_data = generate_inventory_data(warehouses, parent_asin_list)

In [13]:
inventory_data

[{'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B08356M6N7',
  'total_quantity': 62,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B06X3WGFGK',
  'total_quantity': 70,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09Q29B8MG',
  'total_quantity': 33,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B081SPHPCL',
  'total_quantity': 57,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B007136D32',
  'total_quantity': 93,
  

In [14]:
1000*0.75*6*0.99

4455.0

In [15]:
len(inventory_data)

4504

### Write synthetic data to Postgres

In [16]:
def insert_inventory_to_db(inventory_records):
   
    try:
        # Connect to the database
        conn = psycopg2.connect(
            host="localhost",
            port=5433,
            database="tools_database",
            user="tools_user",
            password="tools_user_password"
        )
        conn.autocommit = True

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
            # Prepare the INSERT query
            insert_query = """
            INSERT INTO warehouses.inventory 
            (warehouse_id, warehouse_location, warehouse_name, product_id, total_quantity, reserved_quantity)
            VALUES (%(warehouse_id)s, %(warehouse_location)s, %(warehouse_name)s, %(product_id)s, %(total_quantity)s, %(reserved_quantity)s)
            """
            
            # Use execute_batch for better performance with many inserts
            execute_batch(cursor, insert_query, inventory_records, page_size=100)
            
            # Commit the transaction
            conn.commit()
            
            print(f"Successfully inserted {len(inventory_records)} records into warehouses.inventory")
            
            # Close cursor and connection
            cursor.close()
            conn.close()
        
    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

In [17]:
insert_inventory_to_db(inventory_data)

Successfully inserted 4504 records into warehouses.inventory
